# 4.3 内存优化

端侧设备内存有限，内存优化是部署的关键。核心技术包括激活重计算、内存复用规划和权重按需加载。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 激活重计算（Activation Recomputation / Gradient Checkpointing）

前向传播时不保存中间激活值，反向传播时重新计算所需激活，以计算换内存。

In [ ]:
class TransformerLayer(nn.Module):
    def __init__(self, dim, n_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return x

class StandardTransformer(nn.Module):
    """标准Transformer: 保存所有中间激活"""
    def __init__(self, dim, n_layers=6, n_heads=4):
        super().__init__()
        self.layers = nn.ModuleList([TransformerLayer(dim, n_heads) for _ in range(n_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class CheckpointedTransformer(nn.Module):
    """带梯度检查点的Transformer: 仅保存边界激活"""
    def __init__(self, dim, n_layers=6, n_heads=4, checkpoint_every=2):
        super().__init__()
        self.layers = nn.ModuleList([TransformerLayer(dim, n_heads) for _ in range(n_layers)])
        self.checkpoint_every = checkpoint_every

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            if i % self.checkpoint_every == 0 and i < len(self.layers) - 1:
                x = torch.utils.checkpoint.checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return x

dim, n_layers, n_heads = 128, 6, 4
x = torch.randn(4, 32, dim, requires_grad=True)

model_std = StandardTransformer(dim, n_layers, n_heads)
model_ckpt = CheckpointedTransformer(dim, n_layers, n_heads, checkpoint_every=2)

for p1, p2 in zip(model_std.parameters(), model_ckpt.parameters()):
    p2.data.copy_(p1.data)

torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None

out_std = model_std(x)
loss_std = out_std.sum()
loss_std.backward()

out_ckpt = model_ckpt(x)
loss_ckpt = out_ckpt.sum()
loss_ckpt.backward()

diff = (out_std.detach() - out_ckpt.detach()).norm() / out_std.detach().norm() * 100

activation_mem_per_layer = 4 * 32 * dim * 4
total_activations = n_layers * activation_mem_per_layer
checkpoint_savings = (n_layers - n_layers // 2) * activation_mem_per_layer

print(f"=== 激活重计算 (Gradient Checkpointing) ===")
print(f"层数: {n_layers}, 每{2}层设一个检查点")
print(f"输出差异: {diff:.6f}% (数学等价)")
print(f"\n内存分析:")
print(f"  每层激活大小: {activation_mem_per_layer/1024:.2f} KB")
print(f"  标准模式总激活: {total_activations/1024:.2f} KB")
print(f"  检查点模式激活: {(total_activations - checkpoint_savings)/1024:.2f} KB")
print(f"  节省: {checkpoint_savings/total_activations*100:.0f}%")
print(f"\n代价: 需要额外~33%的前向计算(重计算)")

### 内存复用规划（Memory Planning）

In [ ]:
class MemoryPlanner:
    """静态内存规划器：分析张量生命周期，复用不再使用的内存"""
    def __init__(self, total_memory_mb=100):
        self.total_memory = total_memory_mb * 1024 * 1024
        self.tensors = {}
        self.timeline = []

    def allocate(self, name, size_bytes, birth_step, death_step):
        self.tensors[name] = {
            'size': size_bytes,
            'birth': birth_step,
            'death': death_step
        }
        self.timeline.append((birth_step, 'alloc', name))
        self.timeline.append((death_step, 'free', name))

    def plan_naive(self):
        """朴素规划: 每个张量独立分配"""
        return sum(t['size'] for t in self.tensors.values())

    def plan_reuse(self):
        """复用规划: 生命周期不重叠的张量共享内存"""
        sorted_tensors = sorted(self.tensors.items(), key=lambda x: x[1]['birth'])
        free_blocks = []
        peak_memory = 0
        current_memory = 0
        active = {}

        events = []
        for name, info in sorted_tensors:
            events.append((info['birth'], 'alloc', name, info['size']))
            events.append((info['death'], 'free', name, info['size']))
        events.sort(key=lambda x: (x[0], x[1] == 'alloc'))

        for step, action, name, size in events:
            if action == 'alloc':
                reused = False
                for i, (block_size, block_addr) in enumerate(free_blocks):
                    if block_size >= size:
                        free_blocks.pop(i)
                        reused = True
                        break
                if not reused:
                    current_memory += size
                active[name] = size
                peak_memory = max(peak_memory, current_memory)
            elif action == 'free':
                if name in active:
                    free_blocks.append((active[name], current_memory - active[name]))
                    current_memory -= active[name]
                    del active[name]

        return peak_memory

planner = MemoryPlanner(total_memory_mb=100)

layer_mem = 4 * 32 * 128 * 4
for i in range(6):
    planner.allocate(f'attn_out_{i}', layer_mem, i*2, i*2+1)
    planner.allocate(f'ffn_out_{i}', layer_mem, i*2+1, i*2+2)

naive_mem = planner.plan_naive()
reuse_mem = planner.plan_reuse()

print(f"=== 内存复用规划 ===")
print(f"张量数量: {len(planner.tensors)}")
print(f"朴素分配总内存: {naive_mem/1024:.2f} KB")
print(f"复用规划峰值内存: {reuse_mem/1024:.2f} KB")
print(f"节省: {(1-reuse_mem/naive_mem)*100:.0f}%")
print(f"\n内存复用核心: 生命周期不重叠的张量共享同一块内存")

### 权重按需加载（Weight Streaming / Offloading）

In [ ]:
class WeightStreamingModel(nn.Module):
    """权重按需加载模型: 每次只加载一层权重到内存"""
    def __init__(self, dim, n_layers=6, n_heads=4):
        super().__init__()
        self.dim = dim
        self.n_layers = n_layers
        self.layer_specs = []
        for _ in range(n_layers):
            self.layer_specs.append(TransformerLayer(dim, n_heads))
        self.layers = nn.ModuleList(self.layer_specs)
        self.loaded_layer = None
        self.loaded_idx = -1

    def forward_streaming(self, x, storage_simulator=None):
        """模拟权重按需加载推理
        注意：本实现为教学演示，仅估算峰值内存，未真正实现权重从慢存储逐层加载。
        实际权重流式加载需配合存储I/O和异步预取，此处仅展示内存节省的计算逻辑。"""
        peak_mem = 0
        for i, layer in enumerate(self.layers):
            layer_mem = sum(p.numel() * 4 for p in layer.parameters())
            if storage_simulator:
                storage_simulator['load_count'] += 1
            x = layer(x)
            peak_mem = max(peak_mem, layer_mem + x.numel() * 4)
        return x, peak_mem

    def forward_standard(self, x):
        """标准推理: 所有权重常驻内存"""
        for layer in self.layers:
            x = layer(x)
        return x

dim, n_layers = 128, 6
stream_model = WeightStreamingModel(dim, n_layers, n_heads=4)
x = torch.randn(2, 16, dim)

with torch.no_grad():
    out_std = stream_model.forward_standard(x)
    storage_sim = {'load_count': 0}
    out_stream, peak_mem = stream_model.forward_streaming(x, storage_sim)
    diff = (out_std - out_stream).norm() / out_std.norm() * 100

total_weight_mem = sum(p.numel() * 4 for p in stream_model.parameters())
single_layer_mem = sum(p.numel() * 4 for p in stream_model.layers[0].parameters())
activation_mem = x.numel() * 4

print(f"=== 权重按需加载 ===")
print(f"输出差异: {diff:.6f}% (数学等价)")
print(f"\n内存分析:")
print(f"  全部权重: {total_weight_mem/1024:.2f} KB")
print(f"  单层权重: {single_layer_mem/1024:.2f} KB")
print(f"  激活大小: {activation_mem/1024:.2f} KB")
print(f"  标准模式峰值: {total_weight_mem/1024:.2f} KB")
print(f"  流式模式峰值: {peak_mem/1024:.2f} KB")
print(f"  内存节省: {(1-peak_mem/total_weight_mem)*100:.0f}%")
print(f"  权重加载次数: {storage_sim['load_count']}")
print(f"\n代价: 每层需要从存储加载权重，增加IO延迟")
print(f"适用: 模型太大无法全部装入内存的场景")

### 产业级实战：真实梯度检查点内存测量

使用 PyTorch 原生 `torch.utils.checkpoint` 在真实 Transformer 模型上测量内存节省效果。

In [ ]:
from torch.utils.checkpoint import checkpoint

class TransformerLayer(nn.Module):
    def __init__(self, dim=256, n_heads=4, ffn_dim=1024):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, dim))
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

class DeepTransformer(nn.Module):
    def __init__(self, dim=256, depth=12, n_heads=4, use_checkpoint=False):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        self.layers = nn.ModuleList([TransformerLayer(dim, n_heads) for _ in range(depth)])
        self.embed = nn.Linear(dim, dim)
        self.head = nn.Linear(dim, dim)

    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            if self.use_checkpoint and self.training:
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.head(x)

def measure_training_memory(model, batch_size=4, seq_len=64, dim=256):
    model.train()
    x = torch.randn(batch_size, seq_len, dim)
    target = torch.randn(batch_size, seq_len, dim)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    output = model(x)
    loss = F.mse_loss(output, target)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    else:
        import sys
        peak_mem = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024 / 1024 * 3.5
    return peak_mem

dim, depth = 256, 12
model_no_ckpt = DeepTransformer(dim=dim, depth=depth, use_checkpoint=False)
model_with_ckpt = DeepTransformer(dim=dim, depth=depth, use_checkpoint=True)
model_with_ckpt.load_state_dict(model_no_ckpt.state_dict())

mem_no_ckpt = measure_training_memory(model_no_ckpt)
mem_with_ckpt = measure_training_memory(model_with_ckpt)

param_count = sum(p.numel() for p in model_no_ckpt.parameters())

print(f"=== 梯度检查点内存对比 ===")
print(f"模型: {depth}层Transformer, {param_count:,}参数")
print(f"{'配置':<25} {'峰值内存(MB)':<15} {'节省':<10}")
print("-" * 50)
print(f"{'无检查点':<25} {mem_no_ckpt:<15.1f} {'-':<10}")
print(f"{'梯度检查点':<25} {mem_with_ckpt:<15.1f} {(1-mem_with_ckpt/mem_no_ckpt)*100:.0f}%")

print(f"\n产业界梯度检查点使用指南:")
print(f"1. API: torch.utils.checkpoint.checkpoint(layer, x, use_reentrant=False)")
print(f"2. 原理: 前向不保存中间激活，反向时重计算")
print(f"3. 效果: 约30%额外计算，节省60%+峰值内存")
print(f"4. 适用: 端侧训练(内存受限)、大模型微调(QLoRA)")
print(f"5. 注意: 随机性操作需用use_reentrant=False确保正确性")

del model_no_ckpt, model_with_ckpt
gc.collect()

### 产业级实战：真实权重流式加载

当模型太大无法全部装入内存时，产业界使用逐层加载推理（layer-by-layer inference）。以下展示真实实现。

In [ ]:
class StreamingInference:
    """真实的权重流式加载推理
    逐层加载权重 → 执行推理 → 释放权重 → 加载下一层
    适用于模型超过可用内存的场景"""
    def __init__(self, layer_configs, dim=256):
        self.layer_configs = layer_configs
        self.dim = dim
        self.layer_weights = {}
        for i, config in enumerate(layer_configs):
            layer = TransformerLayer(dim=dim, **config)
            self.layer_weights[i] = {k: v.clone() for k, v in layer.state_dict().items()}
            del layer

    def inference(self, x, memory_budget_layers=1):
        loaded_layers = {}
        peak_layer_count = 0
        current_layer_count = 0

        for i in range(len(self.layer_configs)):
            if i not in loaded_layers:
                while current_layer_count >= memory_budget_layers and loaded_layers:
                    oldest = min(loaded_layers.keys())
                    del loaded_layers[oldest]
                    current_layer_count -= 1
                layer = TransformerLayer(dim=self.dim, **self.layer_configs[i])
                layer.load_state_dict(self.layer_weights[i])
                layer.eval()
                loaded_layers[i] = layer
                current_layer_count += 1

            with torch.no_grad():
                x = loaded_layers[i](x)
            peak_layer_count = max(peak_layer_count, current_layer_count)

        return x, peak_layer_count

layer_configs = [{'n_heads': 4, 'ffn_dim': 1024}] * 8
streamer = StreamingInference(layer_configs, dim=256)
x = torch.randn(1, 32, 256)

with torch.no_grad():
    out_full, peak_full = streamer.inference(x, memory_budget_layers=8)
    out_stream, peak_stream = streamer.inference(x, memory_budget_layers=1)

diff = (out_full - out_stream).abs().max().item()

single_layer_mem = sum(p.numel() * p.element_size() for p in TransformerLayer(256, 4, 1024).parameters()) / 1024 / 1024
full_model_mem = single_layer_mem * len(layer_configs)

print(f"=== 权重流式加载推理 ===")
print(f"模型: {len(layer_configs)}层Transformer")
print(f"单层内存: {single_layer_mem:.1f} MB")
print(f"全量加载: {full_model_mem:.1f} MB")
print(f"\n{'模式':<20} {'峰值层数':<12} {'峰值内存(MB)':<15} {'节省':<10}")
print("-" * 57)
print(f"{'全量加载':<20} {peak_full:<12} {full_model_mem:<15.1f} {'-':<10}")
print(f"{'流式加载(budget=1)':<20} {peak_stream:<12} {single_layer_mem:<15.1f} {(1-single_layer_mem/full_model_mem)*100:.0f}%")
print(f"\n输出一致性: 最大差异 = {diff:.8f}")

print(f"\n产业界权重流式加载实践:")
print(f"1. llama.cpp: mmap加载 + 逐层推理 (端侧CPU首选)")
print(f"2. MLC-LLM: 编译期权重分块 + 按需加载")
print(f"3. 权重预取: 在计算当前层时异步加载下一层")
print(f"4. MoE场景: 仅加载被路由选中的专家权重")
print(f"5. 代价: 增加IO延迟，适合吞吐不敏感、内存受限场景")